# revT2V — serving + demo (Colab)

Pulls a checkpoint trained on Kaggle from the Hub, serves it behind a small FastAPI app, and exposes it publicly through a free Cloudflare quick tunnel so you (or anyone with the link) can hit it from outside Colab — no ngrok account needed.

**Before running:** add your Hugging Face token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar -> add new secret, and toggle notebook access on).

## 1. Clone/pull the repo and install it

In [ ]:
!git clone -q https://github.com/silentlooop/revT2V.git /content/revT2V 2>/dev/null || git -C /content/revT2V pull -q
%cd /content/revT2V
!pip install -q -e .

## 2. Load the HF token and pick which checkpoint to serve

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["REVT2V_HF_REPO"] = "your-username/revt2v-checkpoints"  # TODO: replace
os.environ["REVT2V_DEFAULT_METHOD"] = "baseline"  # TODO: match what you trained

## 3. Start the model server in the background

`scripts/serve.py` loads the teacher + student lazily on the first request per method, so this cell returns immediately — the first `/generate` call will be slower than the rest.

In [ ]:
import subprocess, time

server = subprocess.Popen(
    ["python", "scripts/serve.py"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
time.sleep(5)  # give uvicorn a moment to bind the port
print(server.poll(), "None means still running (good)")

## 4. Download `cloudflared` and open a quick tunnel

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import re, subprocess, time

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(1)
        continue
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("Public URL:", public_url)

## 5. Hit the API

In [ ]:
import requests

print(requests.get(f"{public_url}/health").json())

In [ ]:
job = requests.post(
    f"{public_url}/generate",
    json={"prompt": "a person is jogging", "method": "baseline", "seed": 0},
).json()
print(job)

In [ ]:
import time

job_id = job["job_id"]
while True:
    status = requests.get(f"{public_url}/jobs/{job_id}").json()
    print(status)
    if status["status"] in ("done", "error"):
        break
    time.sleep(5)

## 6. Preview the result

In [ ]:
from IPython.display import Video

Video(f"{public_url}{status['video_url']}") if status["status"] == "done" else status